### **Transformers para traducción de lenguaje**


#### **Configuración**



In [ ]:
# Instalación mínima (sin torchtext ni spaCy)
#!pip install -Uq nltk==3.8.1
#!pip install -Uq matplotlib tqdm
#!pip install -Uq pdfplumber==0.9.0 fpdf==1.7.2

# Recursos usados más adelante en el cuaderno
!wget -nc 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0201EN-Coursera/transformer.pt'
!wget -nc 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0201EN-Coursera/input_de.pdf'


#### **Importación de librerías requeridas**


In [ ]:
import os
import re
import tarfile
import urllib.request
from collections import Counter
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Sequence, Tuple

import matplotlib.pyplot as plt
from nltk.translate.bleu_score import sentence_bleu
import torch
from torch import Tensor
import torch.nn as nn
from torch.nn import Transformer
from torch.utils.data import DataLoader, Dataset
import math
from tqdm import tqdm

# También puedes usar esta sección para suprimir advertencias generadas por tu código:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')


In [ ]:
# Define símbolos especiales e índices
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
# Asegúrate de que los tokens estén en el orden de sus índices para insertarlos correctamente en el vocabulario
special_symbols = ['<unk>', '<pad>', '<bos>', '<eos>']

#### **Cargador de datos (dataLoader)**

En el conjunto de datos Multi30K inglés-alemán, primero cargas los datos y descompones las oraciones en palabras o fragmentos más pequeños, llamados tokens. A partir de estos tokens sea crea una lista única o vocabulario. Luego, cada token se convierte en un número específico usando ese vocabulario. Dado que las oraciones pueden tener longitudes diferentes, se añade **padding** para igualarlas al mismo tamaño en un batch. 

Todos estos datos procesados se organizan en un `DataLoader` de PyTorch, lo que facilita su uso para entrenar redes neuronales.


> En esta versión, el `DataLoader` se implementa manualmente con **Python puro + PyTorch**, sin `torchtext`.

In [ ]:
# Implementación del cargador Multi30K en Python puro + PyTorch

DATA_ROOT = Path("multi30k_data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

MULTI30K_URLS = {
    "train": "https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw/train.tar.gz",
    "valid": "https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw/val.tar.gz",
    "test": "https://raw.githubusercontent.com/multi30k/dataset/master/data/task1/raw/mmt16_task1_test.tar.gz",
}

SRC_LANGUAGE = "de"
TGT_LANGUAGE = "en"

def simple_tokenizer(text: str) -> List[str]:
    text = text.strip().lower()
    # separa palabras y puntuación básica; funciona razonablemente para alemán/inglés
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)

class SimpleVocab:
    def __init__(self, counter: Counter, specials: Sequence[str]):
        self.itos = list(specials)
        seen = set(self.itos)
        for token, _ in counter.most_common():
            if token not in seen:
                self.itos.append(token)
                seen.add(token)
        self.stoi = {token: idx for idx, token in enumerate(self.itos)}
        self.default_index = self.stoi["<unk>"]

    def __len__(self) -> int:
        return len(self.itos)

    def __call__(self, tokens: Sequence[str]) -> List[int]:
        return [self.stoi.get(token, self.default_index) for token in tokens]

    def lookup_tokens(self, indices: Sequence[int]) -> List[str]:
        tokens = []
        for idx in indices:
            idx = int(idx)
            if 0 <= idx < len(self.itos):
                tokens.append(self.itos[idx])
            else:
                tokens.append("<unk>")
        return tokens

def download_and_extract_multi30k(root: Path = DATA_ROOT) -> None:
    for split, url in MULTI30K_URLS.items():
        archive_path = root / f"{split}.tar.gz"
        split_dir = root / split
        split_dir.mkdir(parents=True, exist_ok=True)

        if not archive_path.exists():
            print(f"Descargando {split} desde {url} ...")
            urllib.request.urlretrieve(url, archive_path)

        # Extrae solo si aún no existen archivos .de/.en para el split
        extracted_lang_files = list(split_dir.rglob("*.de")) + list(split_dir.rglob("*.en"))
        if not extracted_lang_files:
            print(f"Extrayendo {archive_path.name} ...")
            with tarfile.open(archive_path, "r:gz") as tar:
                tar.extractall(split_dir)

def _find_parallel_files(split_dir: Path, split_name: str) -> Tuple[Path, Path]:
    de_candidates = sorted(split_dir.rglob("*.de"))
    en_candidates = sorted(split_dir.rglob("*.en"))

    if not de_candidates or not en_candidates:
        raise FileNotFoundError(
            f"No se encontraron archivos .de/.en dentro de {split_dir}. "
            "Verifica la descarga y extracción del dataset."
        )

    # Prioriza archivos cuyo nombre contiene el nombre del split (train/val/test)
    def pick(candidates: List[Path]) -> Path:
        for candidate in candidates:
            name = candidate.name.lower()
            if split_name in name or split_name[:3] in name:
                return candidate
        return candidates[0]

    return pick(de_candidates), pick(en_candidates)

def load_parallel_sentences(split: str, root: Path = DATA_ROOT) -> List[Tuple[str, str]]:
    split_to_dir = {
        "train": root / "train",
        "valid": root / "valid",
        "test": root / "test",
    }
    split_dir = split_to_dir[split]
    de_file, en_file = _find_parallel_files(split_dir, "val" if split == "valid" else split)

    with open(de_file, "r", encoding="utf-8") as f_de, open(en_file, "r", encoding="utf-8") as f_en:
        de_lines = [line.strip() for line in f_de if line.strip()]
        en_lines = [line.strip() for line in f_en if line.strip()]

    paired = list(zip(de_lines, en_lines))
    if not paired:
        raise ValueError(f"No se encontraron pares de oraciones en {split}.")
    return paired

download_and_extract_multi30k()

train_pairs = load_parallel_sentences("train")
valid_pairs = load_parallel_sentences("valid")

def build_vocab_from_pairs(pairs: Sequence[Tuple[str, str]], language: str) -> SimpleVocab:
    idx = 0 if language == SRC_LANGUAGE else 1
    counter = Counter()
    for pair in pairs:
        counter.update(simple_tokenizer(pair[idx]))
    return SimpleVocab(counter, special_symbols)

vocab_transform: Dict[str, SimpleVocab] = {
    SRC_LANGUAGE: build_vocab_from_pairs(train_pairs, SRC_LANGUAGE),
    TGT_LANGUAGE: build_vocab_from_pairs(train_pairs, TGT_LANGUAGE),
}

def sequential_transforms(*transforms):
    def apply_transforms(text_input):
        for transform in transforms:
            text_input = transform(text_input)
        return text_input
    return apply_transforms

def tensor_transform(token_ids: List[int]) -> Tensor:
    return torch.cat(
        (
            torch.tensor([BOS_IDX], dtype=torch.long),
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor([EOS_IDX], dtype=torch.long),
        )
    )

text_transform: Dict[str, Callable[[str], Tensor]] = {
    SRC_LANGUAGE: sequential_transforms(simple_tokenizer, vocab_transform[SRC_LANGUAGE], tensor_transform),
    TGT_LANGUAGE: sequential_transforms(simple_tokenizer, vocab_transform[TGT_LANGUAGE], tensor_transform),
}

class TranslationDataset(Dataset):
    def __init__(self, pairs: Sequence[Tuple[str, str]]):
        self.pairs = list(pairs)

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, idx: int) -> Tuple[str, str]:
        return self.pairs[idx]

def collate_translation_batch(batch: Sequence[Tuple[str, str]]) -> Tuple[Tensor, Tensor]:
    src_batch, tgt_batch = [], []
    for src_sample, tgt_sample in batch:
        src_batch.append(text_transform[SRC_LANGUAGE](src_sample))
        tgt_batch.append(text_transform[TGT_LANGUAGE](tgt_sample))

    src_batch = nn.utils.rnn.pad_sequence(src_batch, padding_value=PAD_IDX)
    tgt_batch = nn.utils.rnn.pad_sequence(tgt_batch, padding_value=PAD_IDX)
    return src_batch, tgt_batch

def get_translation_dataloaders(batch_size: int = 1) -> Tuple[DataLoader, DataLoader]:
    train_dataset = TranslationDataset(train_pairs)
    valid_dataset = TranslationDataset(valid_pairs)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_translation_batch,
    )
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_translation_batch,
    )
    return train_loader, valid_loader

def _tensor_to_sentence(tensor_like, language: str) -> str:
    if isinstance(tensor_like, Tensor):
        values = tensor_like.detach().cpu()
        if values.dim() == 2:
            # si llega [seq_len, batch], usa la primera columna para visualizar un ejemplo
            values = values[:, 0]
        values = values.reshape(-1).tolist()
    elif isinstance(tensor_like, (list, tuple)):
        values = list(tensor_like)
    else:
        values = [int(tensor_like)]

    tokens = vocab_transform[language].lookup_tokens(values)
    return " ".join(tokens)

def index_to_german(tensor_like) -> str:
    return _tensor_to_sentence(tensor_like, SRC_LANGUAGE)

def index_to_eng(tensor_like) -> str:
    return _tensor_to_sentence(tensor_like, TGT_LANGUAGE)

print(f"Tamaño vocabulario ({SRC_LANGUAGE}):", len(vocab_transform[SRC_LANGUAGE]))
print(f"Tamaño vocabulario ({TGT_LANGUAGE}):", len(vocab_transform[TGT_LANGUAGE]))
print("Pares de entrenamiento:", len(train_pairs))
print("Pares de validación:", len(valid_pairs))


Dado el trabajo exploratorio, se usa un tamaño de batch de uno:

In [ ]:
train_dataloader, _ = get_translation_dataloaders(batch_size = 1)

Inicializa un iterador para el dataloader de validación:


In [ ]:
data_itr=iter(train_dataloader)
data_itr

Para obtener ejemplos diversos, se puede iterar por múltiples muestras ya que el dataset está ordenado por longitud:



In [ ]:
for n in range(1000):
    german, english= next(data_itr)

Para compatibilidad con las funciones auxiliares, se transpone los tensores:



In [ ]:
german=german.T
english=english.T

Se imprime el texto convirtiendo los índices a palabras usando `index_to_german` e `index_to_english`:



In [ ]:
for n in range(10):
    german, english= next(data_itr)

    print("Muestra {}".format(n))
    print("Entrada en alemán")
    print(index_to_german(german))
    print("Objetivo en inglés")
    print(index_to_eng(english))
    print("_________\n")

Se define el dispositivo (CPU o GPU) para entrenamiento. 

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

### **Conceptos importantes**


#### **Enmascaramiento (masking)**

Durante el entrenamiento, toda la secuencia está visible para el modelo y se utiliza como entrada para aprender patrones. En cambio, durante la predicción no se dispone de la parte futura de la secuencia. Para simular esta carencia de datos futuros, se emplea el **enmascaramiento (masking)**, garantizando que el modelo aprenda a predecir sin ver los tokens siguientes reales. Esto es crucial para asegurar que ciertas posiciones no sean atendidas. 

La función `generate_square_subsequent_mask` produce una matriz triangular superior, que impide que, durante la decodificación, un token atienda a tokens futuros.

In [ ]:
def generate_square_subsequent_mask(sz,device=DEVICE):
    mask = (torch.triu(torch.ones((sz, sz), device=device)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

Por otro lado, la función `create_mask` genera tanto las máscaras de origen (source) y destino (target) como las máscaras de relleno (padding) basadas en las secuencias proporcionadas. Las máscaras de padding evitan que el modelo atienda a los tokens de relleno, simplificando la atención.

In [ ]:
def create_mask(src, tgt,device=DEVICE):
    src_seq_len = src.shape[0]
    tgt_seq_len = tgt.shape[0]

    tgt_mask = generate_square_subsequent_mask(tgt_seq_len)
    src_mask = torch.zeros((src_seq_len, src_seq_len),device=DEVICE).type(torch.bool)

    src_padding_mask = (src == PAD_IDX).transpose(0, 1)
    tgt_padding_mask = (tgt == PAD_IDX).transpose(0, 1)
    return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask

#### **Codificación posicional (positional encoding)**

El modelo transformer no tiene conocimiento inherente del orden de los tokens en la secuencia. Para proporcionarle esta información, se añaden codificaciones posicionales a los embeddings de los tokens. Estas codificaciones siguen un patrón fijo basado en su posición en la secuencia.


In [ ]:
## Agrega información posicional a los tokens de entrada
class PositionalEncoding(nn.Module):
    def __init__(self,
                 emb_size: int,
                 dropout: float,
                 maxlen: int = 5000):
        super(PositionalEncoding, self).__init__()
        den = torch.exp(- torch.arange(0, emb_size, 2)* math.log(10000) / emb_size)
        pos = torch.arange(0, maxlen).reshape(maxlen, 1)
        pos_embedding = torch.zeros((maxlen, emb_size))
        pos_embedding[:, 0::2] = torch.sin(pos * den)
        pos_embedding[:, 1::2] = torch.cos(pos * den)
        pos_embedding = pos_embedding.unsqueeze(-2)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer('pos_embedding', pos_embedding)

    def forward(self, token_embedding: Tensor):
        return self.dropout(token_embedding + self.pos_embedding[:token_embedding.size(0), :])

#### **Embedding de tokens (token embedding)**

El embedding de tokens, también conocido como embedding de palabras o representación de palabras, convierte palabras o tokens en vectores numéricos en un espacio vectorial continuo. Cada palabra única en el corpus se representa por un vector de longitud fija cuyos valores reflejan propiedades lingüísticas, como significado, contexto o relaciones con otras palabras.

La clase `TokenEmbedding` siguiente convierte tokens numéricos en embeddings:


In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size: int, emb_size):
        super(TokenEmbedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.emb_size = emb_size

    def forward(self, tokens: Tensor):
        return self.embedding(tokens.long()) * math.sqrt(self.emb_size)

#### **Arquitectura Transformer para traducción de lenguaje**

La traducción de lenguaje con un modelo transformer se basa en una arquitectura encoder-decoder. A continuación se desglosan los componentes esenciales y el procedimiento de entrenamiento para una comprensión clara.

##### **Tokenización y codificación posicional**

Primero, el texto en el idioma origen (secuencia de entrada) se tokeniza, dividiéndolo en palabras o subpalabras. Estos tokens se convierten en representaciones numéricas y, para conservar la información de orden, se les añaden codificaciones posicionales.

##### **Procesamiento del encoder**

Estos tokens numéricos se pasan luego por el encoder, compuesto por varias capas que incluyen mecanismos de self-attention y redes feed-forward. Esta arquitectura permite al transformer procesar la secuencia completa de una vez, a diferencia de modelos secuenciales como LSTMs o GRUs.

##### **Decodificación con teacher forcing**

Durante el entrenamiento, el texto en el idioma destino (secuencia correcta de salida) también se tokeniza y convierte en tokens numéricos. El *teacher forcing* es una técnica de entrenamiento en la que el decoder recibe los tokens reales de destino como entrada. El decoder utiliza tanto la salida del encoder como los tokens generados previamente (comenzando con un token especial de inicio) para predecir el siguiente token de la secuencia.

##### **Generación de salida y cálculo de la pérdida**

El decoder genera la traducción token a token. En cada paso, predice el siguiente token de la secuencia destino. La secuencia predicha se compara con la secuencia real usando una función de pérdida, normalmente entropía cruzada, que cuantifica lo bien que coinciden las predicciones del modelo con la secuencia objetivo.

##### **Seq2SeqTransformer**

La clase `Seq2SeqTransformer` representa el núcleo del modelo transformer para traducción de lenguaje. Para entrenarla de manera efectiva, se abordarán:

* **Carga de datos:** Preparar los datos de entrenamiento (texto origen y su traducción).
* **Inicialización del modelo:** Configurar encoder, decoder, codificaciones posicionales y demás componentes.
* **Configuración del optimizador:** Elegir un optimizador (por ejemplo, Adam) y programar la tasa de aprendizaje.
* **Bucle de entrenamiento:** Iterar sobre los datos durante varias épocas, aplicando *teacher forcing*.
* **Monitoreo de la pérdida:** Registrar y, opcionalmente, graficar las pérdidas por época.

In [ ]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self,
                 num_encoder_layers: int,
                 num_decoder_layers: int,
                 emb_size: int,
                 nhead: int,
                 src_vocab_size: int,
                 tgt_vocab_size: int,
                 dim_feedforward: int = 512,
                 dropout: float = 0.1):
        super(Seq2SeqTransformer, self).__init__()

        self.src_tok_emb = TokenEmbedding(src_vocab_size, emb_size)
        self.tgt_tok_emb = TokenEmbedding(tgt_vocab_size, emb_size)
        self.positional_encoding = PositionalEncoding(
            emb_size, dropout=dropout)
        self.transformer = Transformer(d_model=emb_size,
                                       nhead=nhead,
                                       num_encoder_layers=num_encoder_layers,
                                       num_decoder_layers=num_decoder_layers,
                                       dim_feedforward=dim_feedforward,
                                       dropout=dropout)
        self.generator = nn.Linear(emb_size, tgt_vocab_size)

    def forward(self,
                src: Tensor,
                trg: Tensor,
                src_mask: Tensor,
                tgt_mask: Tensor,
                src_padding_mask: Tensor,
                tgt_padding_mask: Tensor,
                memory_key_padding_mask: Tensor):
        src_emb = self.positional_encoding(self.src_tok_emb(src))
        tgt_emb = self.positional_encoding(self.tgt_tok_emb(trg))
        outs = self.transformer(src_emb, tgt_emb, src_mask, tgt_mask, None,
                                src_padding_mask, tgt_padding_mask, memory_key_padding_mask)
        outs =outs.to(DEVICE)
        return self.generator(outs)

    def encode(self, src: Tensor, src_mask: Tensor):
        return self.transformer.encoder(self.positional_encoding(
                            self.src_tok_emb(src)), src_mask)

    def decode(self, tgt: Tensor, memory: Tensor, tgt_mask: Tensor):
        return self.transformer.decoder(self.positional_encoding(
                          self.tgt_tok_emb(tgt)), memory,
                          tgt_mask)

#### **Inferencia**

El siguiente diagrama ilustra el proceso de **inferencia** o **predicción secuencial** en un modelo Transformer encoder-decoder. El proceso comienza alimentando al **encoder**, representado en la sección naranja inferior izquierda, con los índices de la secuencia que se desea traducir.

Los embeddings generados por el encoder se envían al **decoder**, resaltado en verde. Al mismo tiempo, la entrada del decoder inicia con un token especial de comienzo de secuencia, normalmente denotado como `<bos>`, tal como se muestra en la base del bloque verde.

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0201EN-Coursera/predict_transformers.png" alt="transformer" width="600">

La salida del decoder se proyecta mediante una capa lineal hacia un vector cuyo tamaño coincide con el vocabulario del modelo. Luego, una función **softmax** convierte esas puntuaciones en probabilidades sobre todas las palabras posibles.

La palabra con mayor probabilidad se selecciona mediante `argmax`, obteniendo así el índice del token predicho para la traducción. Ese token predicho se retroalimenta al decoder junto con los tokens generados previamente, permitiendo calcular la siguiente palabra de la secuencia.

Este procedimiento se repite de manera **autorregresiva**: en cada paso, el modelo utiliza sus predicciones anteriores como parte de la entrada para generar el siguiente token. En el diagrama, esta retroalimentación se representa mediante la flecha que conecta la salida superior del decoder con su entrada inferior.


In [ ]:
torch.manual_seed(0)

SRC_LANGUAGE = 'de'
TGT_LANGUAGE = 'en'
SRC_VOCAB_SIZE = len(vocab_transform[SRC_LANGUAGE])
TGT_VOCAB_SIZE = len(vocab_transform[TGT_LANGUAGE])
EMB_SIZE = 512
NHEAD = 8
FFN_HID_DIM = 512
BATCH_SIZE = 128
NUM_ENCODER_LAYERS = 3
NUM_DECODER_LAYERS = 3

transformer = Seq2SeqTransformer(NUM_ENCODER_LAYERS, NUM_DECODER_LAYERS, EMB_SIZE,
                                 NHEAD, SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, FFN_HID_DIM)

for p in transformer.parameters():
    if p.dim() > 1:
        nn.init.xavier_uniform_(p)

transformer = transformer.to(DEVICE)

Se carga los pesos del transformer desde el archivo `'transformer.pt'`:

In [ ]:
transformer.load_state_dict(torch.load('transformer.pt', map_location=DEVICE, ))

Dado que el dataset está organizado por longitud de secuencia, se itera para obtener una secuencia más larga:


In [ ]:
for n in range(100):
    src ,tgt= next(data_itr)

In [ ]:
print("engish target",index_to_eng(tgt))
print("german input",index_to_german(src))

Se obtiene el número de tokens de la muestra en alemán:


In [ ]:
num_tokens = src.shape[0]
num_tokens

Se construye una máscara para indicar qué entradas participan en el cálculo de atención. En una tarea de traducción todos los tokens de la secuencia origen están disponibles, por lo que la máscara será `False` en todas las posiciones:


In [ ]:
src_mask = (torch.zeros(num_tokens, num_tokens)).type(torch.bool).to(DEVICE )
src_mask[0:10]

Se extrae la primera muestra del batch de secuencias a traducir. Aunque ahora parezca redundante, será útil cuando se trabaja con batches más grandes:


In [ ]:
src_=src[:,0].unsqueeze(1)
print(src_.shape)
print(src.shape)

Se alimenta los tokens de la secuencia al transformer junto con la máscara. El resultado, guardado en `memory`, son los embeddings producidos por el encoder:

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0201EN-Coursera/Transformersencoder.png" alt="trasfoemr" width="400">

La **memoria** (memory), que es la salida del encoder, encapsula la secuencia original que queremos traducir y sirve como entrada para el decoder:


In [ ]:
memory = transformer.encode(src_, src_mask)
memory.shape

Para iniciar la generación de la secuencia de salida, comienza con el símbolo de inicio  (`<bos>`):


In [ ]:
ys = torch.ones(1, 1).fill_(BOS_IDX).type(torch.long).to(DEVICE)
ys

Por convención de nombres, el término **target** se usa para denotar la predicción. En este contexto, "target" se refiere a las palabras que siguen a la predicción actual. 

Estas pueden combinarse con la secuencia de origen para realizar predicciones posteriores. Por tanto, se construye una máscara de destino donde todos los valores son `False`, indicando que no se debe ignorar ninguna posición:


In [ ]:
tgt_mask = (generate_square_subsequent_mask(ys.size(0)).type(torch.bool)).to(DEVICE)
tgt_mask

Se alimenta la salida del encoder (`memory`) y la predicción previa (por ahora solo el token de inicio) al decoder:


In [ ]:
out = transformer.decode(ys, memory, tgt_mask)
out.shape


La salida del decoder es un embedding enriquecido de la palabra predicha. A continuación, se elimina la dimensión de batch reordenando:


In [ ]:
out = out.transpose(0, 1)
out.shape

Una vez que el decoder produce su salida, esta se envía a la capa de salida para obtener los logits sobre el vocabulario completo, compuesto por 10 837 palabras.

En la etapa de inferencia, solo se necesita la predicción correspondiente al último token generado. Por ello, se puede seleccionar la última posición de la secuencia usando: `out[:, -1]`.

In [ ]:
logit = transformer.generator(out[:, -1])
logit.shape

El proceso se ilustra de manera concisa en la imagen siguiente:

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0201EN-Coursera/decoder_start.png" alt="trasfoemr" width="600">


La palabra predicha se determina identificando el valor de logit más alto, lo cual indica la traducción más probable del modelo para la entrada en una posición específica; esta posición corresponde al índice del siguiente token.


In [ ]:
  _, next_word_index = torch.max(logit, dim=1)

In [ ]:
print("Salida en ingles:",index_to_eng(next_word_index))

In [ ]:
next_word_index=next_word_index.item()
next_word_index

Ahora, se concatena la palabra recién predicha a las predicciones anteriores, permitiendo que el modelo considere toda la secuencia de palabras generadas al realizar su próxima predicción.


In [ ]:
ys = torch.cat([ys,torch.ones(1, 1).type_as(src.data).fill_(next_word_index)], dim=0)
ys

Para predecir la siguiente palabra de la traducción, primero se actualiza la **máscara de destino** para asegurar que el modelo solo atienda a los tokens ya generados. Luego, se pasa la secuencia parcial al **decoder** del Transformer, junto con la salida del **encoder**, para obtener los **logits** sobre el vocabulario. Estos logits se convierten en probabilidades y se selecciona como predicción el token con mayor probabilidad.

Ten en cuenta que la salida del **encoder** ya contiene la representación contextual completa de la secuencia de entrada. Por eso, durante la inferencia no es necesario volver a ejecutar el encoder en cada paso, basta con reutilizar esa representación mientras el decoder genera la traducción token por token.



In [ ]:
# Actualiza la máscara de destino para la longitud de la secuencia actual.
tgt_mask = (generate_square_subsequent_mask(ys.size(0)).type(torch.bool)).to(DEVICE)
tgt_mask

In [ ]:
# Decodifica la secuencia actual utilizando el transformer y recupera la salida
out = transformer.decode(ys, memory, tgt_mask)
out = out.transpose(0, 1)
out.shape

In [ ]:
out[:, -1].shape

In [ ]:
# Obtiene las probabilidades de palabras para la última palabra predicha.
prob = transformer.generator(out[:, -1])
# Encuentra el índice de palabra con la probabilidad más alta.
_, next_word_index = torch.max(prob, dim=1)
# Imprime la palabra en inglés predicha.
print("Salida en inglés:", index_to_eng(next_word_index))
# Convierte el valor del tensor a un escalar de Python.
next_word_index = next_word_index.item()


Ahora, se actualiza la predicción.


In [ ]:
ys = torch.cat([ys,torch.ones(1, 1).type_as(src.data).fill_(next_word_index)], dim=0)
print("Salida en inglés",index_to_eng(ys))

El proceso puede resumirse así: a partir de la representación generada por el **encoder** y del token inicial `<BOS>`, el **decoder** genera un token a la vez. En cada paso, el token predicho se agrega a la secuencia parcial y se vuelve a introducir en el decoder para predecir el siguiente token.

Este ciclo continúa hasta que se cumple un criterio de parada, por ejemplo, cuando el modelo genera el token de fin de secuencia `<EOS>` o cuando se alcanza una longitud máxima permitida. En una implementación simple, también puede limitarse la generación haciendo que la longitud de la secuencia traducida no exceda la longitud de la secuencia original.

Como se muestra en la siguiente imagen, este procedimiento se implementa mediante la función `greedy_decode`, que selecciona en cada paso el token con mayor probabilidad.

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0201EN-Coursera/decoder.png" alt="transformer decoder" width="500">


In [ ]:
def greedy_decode(modelo, src, src_mask, max_len, start_symbol):
    src = src.to(DEVICE)
    src_mask = src_mask.to(DEVICE)

    memory = modelo.encode(src, src_mask)
    ys = torch.ones(1, 1).fill_(start_symbol).type(torch.long).to(DEVICE)
    for i in range(max_len-1):
        memory = memory.to(DEVICE)
        tgt_mask = (generate_square_subsequent_mask(ys.size(0))
                    .type(torch.bool)).to(DEVICE)
        out = modelo.decode(ys, memory, tgt_mask)
        out = out.transpose(0, 1)
        prob = modelo.generator(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.item()

        ys = torch.cat([ys,
                        torch.ones(1, 1).type_as(src.data).fill_(next_word)], dim=0)
        if next_word == EOS_IDX:
            break
    return ys

Se recupera los índices para el idioma alemán y genera la máscara correspondiente:

In [ ]:
src
src_mask = (torch.zeros(num_tokens, num_tokens)).type(torch.bool).to(DEVICE )

Establece un valor razonable para la longitud máxima de la secuencia objetivo:


In [ ]:
max_len=src.shape[0]+5
max_len

Se aplica la función `greedy_decode` a los datos:


In [ ]:
ys=greedy_decode(transformer, src, src_mask, max_len, start_symbol=BOS_IDX)
print("Inglés ",index_to_eng(ys))

In [ ]:
print("Inglés ",index_to_eng(tgt))

#### **Entrenamiento vs. inferencia en traducción automática neuronal**

Durante la fase de inferencia, cuando el modelo se despliega para tareas reales de traducción, el decodificador genera la secuencia sin tener acceso a la secuencia objetivo esperada. En su lugar, basa sus predicciones en la salida del codificador y en los tokens que ha generado hasta el momento. 

El proceso es autorregresivo, con el decodificador prediciendo continuamente el siguiente token hasta que produce un token de fin de secuencia, indicando que la traducción está completa.

La principal diferencia entre las etapas de entrenamiento e inferencia radica en las entradas al decodificador. Durante el entrenamiento, el decodificador se beneficia de la exposición de los ground truth, recibiendo los tokens exactos de la secuencia objetivo de manera incremental a través de una técnica conocida como "teacher forcing". 

Este enfoque contrasta marcadamente con algunas otras arquitecturas de redes neuronales que confían en las predicciones anteriores de la propia red como entradas durante el entrenamiento. Una vez finalizado el entrenamiento, los conjuntos de datos utilizados se asemejan a los empleados en modelos de redes neuronales más convencionales, proporcionando una base familiar para la comparación y evaluación.

Primero, importa la pérdida `CrossEntropyLoss` y crea un objeto de pérdida de entropía cruzada. La pérdida **no** se calculará cuando el token con índice `PAD_IDX` sea una entrada.

In [ ]:
from torch.nn import CrossEntropyLoss

loss_fn = CrossEntropyLoss(ignore_index=PAD_IDX)

Se elimina la última muestra del objetivo:


In [ ]:
tgt_input = tgt[:-1, :]
print(index_to_eng(tgt_input))
print(index_to_eng(tgt))

Se crea las máscaras requeridas:

In [ ]:
src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(src, tgt_input)
print(f"Forma de src_mask: {src_mask.shape}")
print(f"Forma de tgt_mask: {tgt_mask.shape}")
print(f"Forma de src_padding_mask: {src_padding_mask.shape}")
print(f"Forma de tgt_padding_mask: {tgt_padding_mask.shape}")

In [ ]:
src_padding_mask

In [ ]:
print(tgt_mask)
[index_to_eng( tgt_input[t==0])  for t in tgt_mask] #index_to_eng(tgt_input))

Cuando se llama a `modelo(src, tgt_input, src_mask, tgt_mask, src_padding_mask, tgt_padding_mask, memory_key_padding_mask)`, se invoca el método **forward** de la clase `Seq2SeqTransformer`. 

Este proceso genera *logits* para la secuencia objetivo, los cuales pueden luego convertirse en tokens reales tomando la predicción de mayor probabilidad en cada paso de la secuencia.


#### **Función de pérdida**


In [ ]:
logits = transformer(src, tgt_input, src_mask, tgt_mask,src_padding_mask, tgt_padding_mask, src_padding_mask)

print("Formato de la salida ",logits.shape)
print("Formato del objetivo",tgt_input.shape)
print("Formato de la fuente ",src.shape)

Durante la fase de **entrenamiento**, la secuencia objetivo cumple dos funciones relacionadas, pero no idénticas. Por un lado, se utiliza para construir la **entrada del decoder**; por otro, sirve como referencia para calcular la pérdida y medir qué tan correcta es la predicción del modelo.

Para evitar confusiones, llamaremos **entrada del decoder** a la versión desplazada de la secuencia objetivo que se entrega al decoder. Esta secuencia normalmente comienza con el token especial `<BOS>` y contiene los tokens anteriores que el modelo debe usar como contexto.

En cambio, llamaremos **objetivo de salida** a la secuencia que el modelo debe predecir. Esta corresponde a la misma traducción, pero desplazada una posición respecto de la entrada del decoder. Es decir, en cada paso, el modelo recibe los tokens previos y debe predecir el siguiente token.

Por ejemplo:

```text
Entrada del decoder:  <BOS>  I     am    here
Objetivo de salida:   I      am    here  <EOS>
```

De esta manera, el entrenamiento respeta la naturaleza **autorregresiva** del modelo: en cada posición, el decoder aprende a predecir el siguiente token usando únicamente los tokens anteriores y la representación generada por el encoder.



In [ ]:
tgt_out = tgt[1:, :]
print(tgt_out.shape)
[index_to_eng(t)  for t in tgt_out]

Los índices de los tokens representan las clases a predecir. Al aplanar el tensor, cada índice se convierte en una muestra distinta, sirviendo como objetivo para la pérdida de entropía cruzada.


In [ ]:
tgt_out_flattened = tgt_out.reshape(-1)
print(tgt_out_flattened.shape)
tgt_out_flattened

En este modelo **autorregresivo**, se pueden mostrar los tokens de la **entrada del objetivo** después de aplicar la **máscara causal**. Junto a ellos, conviene mostrar también el **objetivo de salida**, para visualizar qué token debe predecir el modelo en cada posición.

La máscara impide que el decoder atienda a tokens futuros. Por ello, en cada paso, el modelo solo puede utilizar los tokens anteriores de la secuencia objetivo, junto con la representación producida por el encoder, para predecir el siguiente token.

Esta visualización permite entender mejor la lógica del entrenamiento autorregresivo: el modelo no predice valores pasados a partir de los presentes, sino que aprende a predecir el **siguiente token** a partir del contexto ya disponible.


In [ ]:
["Entrada: {} objetivo: {}".format(index_to_eng( tgt_input[m==0]),index_to_eng( t))  for m,t in zip(tgt_mask,tgt_out)] 

Ahora se calcula la **pérdida** comparando la salida del **decoder** del Transformer con la secuencia objetivo correspondiente. Para ello, los **logits** producidos por el modelo se entregan a la función de pérdida de **entropía cruzada**, junto con los tokens reales que el modelo debe predecir.

La salida del Transformer suele tener tres dimensiones: **longitud de la secuencia**, **tamaño del lote** y **tamaño del vocabulario**. Esta última dimensión corresponde a los logits asociados a cada palabra posible del vocabulario.

Sin embargo, la función de entropía cruzada espera que las predicciones estén organizadas como una colección de decisiones independientes, una por cada token de cada secuencia del lote. Por esta razón, es necesario **reorganizar** o **remodelar** la salida del modelo para que todas las posiciones temporales y todos los ejemplos del lote queden alineados en una misma dimensión.

Este paso permite calcular la pérdida correctamente, comparando en cada posición de la secuencia los logits predichos por el modelo con el **token objetivo real**. Así, la función de pérdida evalúa qué tan bien el Transformer predice el siguiente token en cada paso de tiempo y para cada ejemplo del lote.



In [ ]:
loss = loss_fn(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
print(loss)

#### **El cálculo de la función pérdida**


In [ ]:
# logits.reshape(-1, logits.shape[-1]) transforma el tensor logits en un tensor 2D con forma [longitud_de_secuencia * tamaño_de_lote, tamaño_del_vocabulario]. Este cambio de forma se realiza para alinear tanto los logits predichos como las salidas objetivo para el cálculo de la pérdida.
print("La forma de logits es:", logits.shape)
logits_flattened = logits.reshape(-1, logits.shape[-1])
print("La forma de logit_flats es:", logits_flattened.shape)

# tgt_out.reshape(-1) convierte el tensor tgt_out en un tensor 1D al aplanarlo a lo largo de las dimensiones de secuencia y tamaño de lote. Esto se hace para alinearlo con los logits transformados.
print("La forma de tgt_outs es:", tgt_out.shape)
tgt_out_flattened = tgt_out.reshape(-1)
print("La forma de tgt_out_flats es:", tgt_out_flattened.shape)


Dentro de la función de pérdida, los logits se transformarán en probabilidades entre `[0,1]` que suman 1:

In [ ]:
# Aplicando la función de pérdida de la entropia cruzada
probs = torch.nn.functional.softmax(logits_flattened, dim=1)
probs[1].sum()

Verifica las probabilidades para algunos tokens aleatorios:


In [ ]:
### Tu respuesta

In [ ]:
for i in range (5):
    # using argmax, you can retrieve the index of the token that is predicted with the highest probaility
    print("Id del token predicho:",probs[i].argmax().item(), "probabilidad predicha:",probs[i].max().item())
    # you can also check the actual token from the tgt_out_flat
    print("Id del token actual:",tgt_out_flattened[i].item(), "probabilidad predicha:", probs[i,tgt_out_flattened[i]].item(),"\n")

Se calcula la diferencia entre la probabilidad del token real (1) y las probabilidades predichas para cada token:

In [ ]:
neg_log_likelihood = torch.nn.functional.nll_loss(probs, tgt_out_flattened)
# Obteniendo el valor de perdida 
loss = neg_log_likelihood

# Imprime el valor total de perdida
print("Función de pérdida:", loss.item())

#### **Evaluación**


In [ ]:
def evaluate(modelo):
    modelo.eval()
    losses = 0

    for src, tgt in val_dataloader:
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)

        tgt_input = tgt[:-1, :]

        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(src, tgt_input)
        logits = modelo(src, tgt_input, src_mask, tgt_mask,src_padding_mask, tgt_padding_mask, src_padding_mask)

        tgt_out = tgt[1:, :]
        loss = loss_fn(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
        losses += loss.item()

    return losses / len(list(val_dataloader))

#### **Entrenando el modelo**


In [ ]:
def train_epoch(modelo, optimizer, train_dataloader):
    modelo.train()
    losses = 0

    # Envuelve train_dataloader con tqdm para registrar el progreso
    train_iterator = tqdm(train_dataloader, desc="Training", leave=False)

    for src, tgt in train_iterator:
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)

        tgt_input = tgt[:-1, :]

        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = create_mask(src, tgt_input)
        src_mask = src_mask.to(DEVICE)
        tgt_mask = tgt_mask.to(DEVICE)
        src_padding_mask = src_padding_mask.to(DEVICE)
        tgt_padding_mask = tgt_padding_mask.to(DEVICE)

        logits = modelo(src, tgt_input, src_mask, tgt_mask, src_padding_mask, tgt_padding_mask, src_padding_mask)
        logits = logits.to(DEVICE)

        optimizer.zero_grad()

        tgt_out = tgt[1:, :]
        loss = loss_fn(logits.reshape(-1, logits.shape[-1]), tgt_out.reshape(-1))
        loss.backward()

        optimizer.step()
        losses += loss.item()

        # Actualiza la barra de progreso de tqdm con la pérdida actual
        train_iterator.set_postfix(loss=loss.item())

    return losses / len(list(train_dataloader))

La configuración para el modelo de traducción incluye un tamaño de vocabulario de origen y destino determinado por los idiomas del conjunto de datos, un tamaño de embedding de 512, 8 cabecera de atención, una dimensión oculta para la red feed-forward de 512 y un tamaño de lote de 128. 

El modelo está estructurado con tres capas tanto en el encoder como en el decoder.



In [ ]:
torch.manual_seed(0)

SRC_VOCAB_SIZE = len(vocab_transform[SRC_LANGUAGE])
TGT_VOCAB_SIZE = len(vocab_transform[TGT_LANGUAGE])
EMB_SIZE = 512
NHEAD = 8
FFN_HID_DIM = 512
BATCH_SIZE = 128
NUM_ENCODER_LAYERS = 3
NUM_DECODER_LAYERS = 3

In [ ]:
train_dataloader, val_dataloader = get_translation_dataloaders(batch_size = BATCH_SIZE)

In [ ]:
transformer = Seq2SeqTransformer(NUM_ENCODER_LAYERS, NUM_DECODER_LAYERS, EMB_SIZE,
                                 NHEAD, SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, FFN_HID_DIM)
transformer = transformer.to(DEVICE)

In [ ]:
optimizer = torch.optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

In [ ]:
TrainLoss=[]
ValLoss=[]

Se entrena el modelo durante 10 épocas usando las funciones anteriores.

>Ten en cuenta que entrenar el modelo usando CPUs puede ser un proceso que lleva mucho tiempo.
>Si no tienes acceso a  GPUs, puede pasar directamente a "cargar el modelo guardado" y proceder a cargar el modelo preentrenado usando el código proporcionado. 

In [ ]:
from timeit import default_timer as timer
NUM_EPOCHS = 10

for epoch in range(1, NUM_EPOCHS+1):
    start_time = timer()
    train_loss = train_epoch(transformer, optimizer, train_dataloader)
    TrainLoss.append(train_loss)
    end_time = timer()
    val_loss = evaluate(transformer)
    ValLoss.append(val_loss)
    print((f"Epoca: {epoch}, Pérdida de entrenamiento: {train_loss:.3f}, Pérdida de validación: {val_loss:.3f}, "f"Tiempo de epocas = {(end_time - start_time):.3f}s"))
torch.save(transformer.state_dict(), 'transformer_de_to_en_model.pt')

In [ ]:
epochs = range(1, len(TrainLoss) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs, TrainLoss, 'r', label='Pérdida de entrenamiento')
plt.plot(epochs,ValLoss, 'b', label='Pérdida de validación')
plt.title('Pérdida de entrenamiento y validación')
plt.xlabel('Epocas')
plt.ylabel('Pérdida')
plt.legend()
plt.show()

#### **Carga del modelo guardado**


In [ ]:
#!wget 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0201EN-Coursera/transformer_de_to_en_model.pt'
#transformer.load_state_dict(torch.load('transformer_de_to_en_model.pt',map_location=torch.device('cpu')))

#### **Traducción y evaluación**

Usando la función `greedy_decode` se puede crear una función traductor que genere la traducción al inglés de un texto alemán de entrada.


In [ ]:
# Traduce la frase de entrada al idioma objetivo
def translate(modelo: torch.nn.Module, src_sentence: str):
    modelo.eval()
    src = text_transform[SRC_LANGUAGE](src_sentence).view(-1, 1)
    num_tokens = src.shape[0]
    src_mask = (torch.zeros(num_tokens, num_tokens)).type(torch.bool)
    tgt_tokens = greedy_decode(
        modelo,  src, src_mask, max_len=num_tokens + 5, start_symbol=BOS_IDX).flatten()
    return " ".join(vocab_transform[TGT_LANGUAGE].lookup_tokens(list(tgt_tokens.cpu().numpy()))).replace("<bos>", "").replace("<eos>", "")

In [ ]:
for n in range(5):
    german, english = next(data_itr)

    print("Oración en alemán:", index_to_german(german).replace("<bos>", "").replace("<eos>", ""))
    print("Traducción al inglés:", index_to_eng(english).replace("<bos>", "").replace("<eos>", ""))
    print("Traducción del modelo:", translate(transformer, index_to_german(german)))

#### **Evaluación con la puntuación BLEU**

Para evaluar las traducciones generadas, se introduce la función `calculate_bleu_score`, encargada de calcular la puntuación **BLEU**. Esta métrica se utiliza comúnmente en traducción automática para comparar una traducción generada por el modelo con una o más traducciones de referencia.

La puntuación BLEU proporciona una medida cuantitativa de la calidad de la traducción, evaluando el grado de coincidencia entre los fragmentos de texto generados y los textos de referencia. En términos generales, una puntuación BLEU más alta indica una mayor similitud con las traducciones esperadas.

El código también incluye un ejemplo práctico de cómo aplicar esta función para calcular la puntuación BLEU de una traducción producida por el modelo.


In [ ]:
def calculate_bleu_score(generated_translation, reference_translations):
    # convierte las traducciones generadas y las traducciones de referencia al formato esperado por sentence_bleu
    references = [reference.split() for reference in reference_translations]
    hypothesis = generated_translation.split()

    # calcula la puntuación BLEU
    bleu_score = sentence_bleu(references, hypothesis)

    return bleu_score

In [ ]:
generated_translation = translate(transformer,"Ein brauner Hund spielt im Schnee .")

reference_translations = [
    "A brown dog is playing in the snow .",
    "A brown dog plays in the snow .",
    "A brown dog is frolicking in the snow .",
    "In the snow, a brown dog is playing ."

]

bleu_score = calculate_bleu_score(generated_translation, reference_translations)
print(" Puntuación de BLEU:", bleu_score, "for",generated_translation)

#### **Ejercicio: Traducir un documento**

En este ejercicio, se pide implementar una función que traduzca un PDF en alemán a inglés. Para ello se debe seguir los siguientes pasos:

1. **Definir la función de traducción**
   Crea una función llamada `translate_pdf` que reciba los siguientes parámetros:

   * `input_file`: La ruta al archivo PDF de entrada que se va a traducir.
   * `translator_model`: Un modelo o función que manejará la traducción del texto.
   * `output_file`: La ruta donde se guardará el PDF traducido.

2. **Leer y traducir el PDF**
   Utiliza `pdfplumber` para abrir y leer el texto de cada página del PDF de entrada. Traduce el texto extraído usando el `translator_model`.

3. **Formatear y escribir el texto traducido en un nuevo PDF**

   * Usa `textwrap` para ajustar el texto traducido de manera que se adapte al ancho de página A4.
   * Crea un nuevo PDF con `FPDF` y añade el texto traducido ajustado.
   * Guarda el nuevo PDF con el texto traducido en `output_file`.


In [ ]:
import pdfplumber
import textwrap
from fpdf import FPDF

def translate_pdf(input_file, translator_model,output_file):
    #Completa la función

In [ ]:
!wget 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0201EN-Coursera/input_de.pdf'

In [ ]:
input_file_path =
output_file = 'output_en.pdf'
# Llama a translate_pdf() definido anteriormente para el archivo de entrada
print("El archivo PDF traducido se guarda como:", output_file)


In [ ]:
## Tus respuestas